# Friend A — leak-free LOEO validator from scratch

This notebook implements the first task assigned to Friend A in `TEAM.md`:

- train **from scratch** on the fixed `6bba` side of
  `experiments/_lib/loeo_train6bba_val44b6.json`;
- use the 12 fixed `44b6` videos only for validation;
- run 5–10 full epochs (default: 5) on a single 16 GB GPU;
- save a self-contained hand-off artifact for Friend B/Vitalii.

The current manifest intentionally contains 126 training videos rather than all 128 local `6bba` videos. The two
excluded IDs are the public dummy-test copies (`6bba_05b6850b` and `6bba_05db0fb1`). The manifest is the source of
truth for this experiment.

Important:

1. Start Jupyter from the repository root and select the Python 3.12 project environment.
2. Run all cells. Training starts automatically and can take several hours.
3. This notebook never loads pilkwang or team checkpoints. The generated command is rejected if it contains
   `--init-weights` or `--unet-weights`.
4. The per-epoch `acc * recall` printed by the training script is only an internal sparse-GT diagnostic. It is **not**
   the official competition score. Friend B performs prediction and official evaluation afterwards.
5. Do not submit anything to the leaderboard from this notebook.

You may override settings before starting Jupyter, for example:

```powershell
$env:BIOHUB_DATA_DIR = "C:\path\to\biohub-cell-tracking-during-development\train"
$env:BIOHUB_FRIEND_A_EPOCHS = "7"
jupyter lab
```


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path


EXPERIMENT = "local_friend_a_exp01_loeo_scratch"
METHOD = "friend_a_scratch_bce_w2"

# Friend A defaults from TEAM.md. Environment variables make the notebook reusable
# for the later controlled variants without editing generated cells.
RUN_TRAINING = os.environ.get("BIOHUB_RUN_TRAINING", "1") != "0"
EPOCHS = int(os.environ.get("BIOHUB_FRIEND_A_EPOCHS", "5"))
BATCH_SIZE = int(os.environ.get("BIOHUB_FRIEND_A_BATCH", "4"))
NUM_WORKERS = int(os.environ.get("BIOHUB_FRIEND_A_WORKERS", "4"))
LEARNING_RATE = float(os.environ.get("BIOHUB_FRIEND_A_LR", "1e-4"))
SEED = int(os.environ.get("BIOHUB_FRIEND_A_SEED", "42"))
WINDOW_SIZE = int(os.environ.get("BIOHUB_FRIEND_A_WINDOW", "2"))
DET_LOSS = os.environ.get("BIOHUB_FRIEND_A_DET_LOSS", "bce")
DET_NEG_WEIGHT = float(os.environ.get("BIOHUB_FRIEND_A_DET_NEG_WEIGHT", "0.01"))
DET_LOSS_WEIGHT = float(os.environ.get("BIOHUB_FRIEND_A_DET_LOSS_WEIGHT", "1.0"))
DOWNSAMPLE = os.environ.get("BIOHUB_FRIEND_A_DOWNSAMPLE", "1,4,4")
POOL_KERNEL_UM = float(os.environ.get("BIOHUB_FRIEND_A_POOL_KERNEL_UM", "5.0"))
AUG_INTENSITY = os.environ.get("BIOHUB_FRIEND_A_AUG_INTENSITY", "0") == "1"
MAX_ITERS_TEXT = os.environ.get("BIOHUB_FRIEND_A_MAX_ITERS", "").strip()
MAX_ITERS = int(MAX_ITERS_TEXT) if MAX_ITERS_TEXT else None

if not 5 <= EPOCHS <= 10:
    raise ValueError("TEAM.md requires 5–10 epochs for the leak-free validator")
if BATCH_SIZE not in (4, 5, 6, 8):
    raise ValueError("Use batch 4–6 without gradient checkpointing, or explicitly choose 8 after enabling it")
if BATCH_SIZE == 8:
    raise ValueError("This notebook disables gradient checkpointing; use batch 4–6 on a 16 GB GPU")
if DET_LOSS not in {"bce", "gauss_focal"}:
    raise ValueError("BIOHUB_FRIEND_A_DET_LOSS must be bce or gauss_focal")


def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "TEAM.md").exists() and (candidate / "tools" / "repo" / "scripts").exists():
            return candidate
    raise FileNotFoundError("Repository root not found. Start Jupyter inside the cloned project.")


PROJECT_ROOT = find_project_root(Path.cwd())

explicit_data = os.environ.get("BIOHUB_DATA_DIR", "").strip()
data_candidates = [
    Path(explicit_data).expanduser() if explicit_data else None,
    PROJECT_ROOT / "biohub-cell-tracking-during-development" / "train",
    PROJECT_ROOT / "train",
    PROJECT_ROOT.parent / "train",  # convenient for a worktree next to the local dataset
]
DATA_DIR = next((path.resolve() for path in data_candidates if path is not None and path.exists()), None)
if DATA_DIR is None:
    checked = "\n".join(str(path) for path in data_candidates if path is not None)
    raise FileNotFoundError(
        "Training data were not found. Set BIOHUB_DATA_DIR to the directory containing *.zarr and *.geff.\n"
        f"Checked:\n{checked}"
    )

RUN_ID = os.environ.get("BIOHUB_RUN_ID", datetime.now().strftime("%Y%m%d_%H%M%S"))
OUTPUT_ROOT = Path(os.environ.get("BIOHUB_OUTPUT_DIR", PROJECT_ROOT / "outputs")).expanduser().resolve()
RUN_DIR = OUTPUT_ROOT / EXPERIMENT / RUN_ID
TRAIN_REPO = RUN_DIR / "train_repo"
ARTIFACT_DIR = RUN_DIR / "artifact"
SPLIT_SOURCE = PROJECT_ROOT / "experiments" / "_lib" / "loeo_train6bba_val44b6.json"
PATCH_SOURCE = PROJECT_ROOT / "experiments" / "_lib" / "train_patches.py"

print(f"project: {PROJECT_ROOT}")
print(f"data:    {DATA_DIR}")
print(f"run:     {RUN_DIR}")
print(
    f"recipe: scratch | epochs={EPOCHS} batch={BATCH_SIZE} workers={NUM_WORKERS} "
    f"lr={LEARNING_RATE:g} window={WINDOW_SIZE} det_loss={DET_LOSS} seed={SEED}"
)


In [ ]:
# Preflight, immutable split conversion, and a writable copy of the training code.

required_modules = ["torch", "tracksdata", "geff", "polars", "rustworkx", "zarr", "tqdm"]
evaluation_modules = ["scipy", "skimage", "dask", "networkx", "pandas", "sklearn", "pyscipopt", "ilpy", "blosc2"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError(
        "Missing packages: " + ", ".join(missing_modules) + "\n"
        "Install the dependencies listed in TEAM.md into the selected Python 3.12 environment."
    )
missing_evaluation_modules = [name for name in evaluation_modules if importlib.util.find_spec(name) is None]
if missing_evaluation_modules:
    print(
        "Warning: optional inference/evaluation packages are missing: "
        + ", ".join(missing_evaluation_modules)
        + ". They do not block training, but TEAM.md requires them before official evaluation."
    )

import torch

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Python 3.12 is required by TEAM.md; current interpreter: {sys.version.split()[0]}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Install a CUDA 12.8+ PyTorch build and restart the notebook kernel.")

# A real allocation catches driver/build mismatches that nvidia-smi alone cannot detect.
probe = torch.zeros(1, device="cuda")
del probe
torch.cuda.empty_cache()
device_name = torch.cuda.get_device_name(0)
device_capability = torch.cuda.get_device_capability(0)
print(
    f"CUDA OK: torch={torch.__version__}, runtime={torch.version.cuda}, "
    f"device={device_name}, capability={device_capability}"
)

split_manifest = json.loads(SPLIT_SOURCE.read_text(encoding="utf-8"))
train_ids = list(split_manifest["train"])
holdout_ids = list(split_manifest["holdout"])

if len(train_ids) != 126 or len(holdout_ids) != 12:
    raise RuntimeError(f"Unexpected LOEO manifest size: train={len(train_ids)}, holdout={len(holdout_ids)}")
if len(train_ids) != len(set(train_ids)) or len(holdout_ids) != len(set(holdout_ids)):
    raise RuntimeError("Duplicate dataset IDs in the LOEO manifest")
if set(train_ids) & set(holdout_ids):
    raise RuntimeError("Leakage: train and holdout IDs overlap")
if any(not dataset_id.startswith("6bba_") for dataset_id in train_ids):
    raise RuntimeError("Leakage: every training ID must belong to embryo 6bba")
if any(not dataset_id.startswith("44b6_") for dataset_id in holdout_ids):
    raise RuntimeError("Leakage: every holdout ID must belong to embryo 44b6")

missing_data = []
for dataset_id in train_ids + holdout_ids:
    for suffix in (".zarr", ".geff"):
        expected = DATA_DIR / f"{dataset_id}{suffix}"
        if not expected.exists():
            missing_data.append(str(expected))
if missing_data:
    preview = "\n".join(missing_data[:20])
    raise FileNotFoundError(f"Missing {len(missing_data)} dataset paths. First entries:\n{preview}")

all_6bba = {path.name.removesuffix(".zarr") for path in DATA_DIR.glob("6bba_*.zarr")}
excluded_6bba = sorted(all_6bba - set(train_ids))
print(f"LOEO split OK: {len(train_ids)} train 6bba | {len(holdout_ids)} validation 44b6")
if excluded_6bba:
    print("6bba videos outside the fixed manifest:", excluded_6bba)

RUN_DIR.mkdir(parents=True, exist_ok=False)
TRAIN_REPO.mkdir(parents=True)
for subdir in ("scripts", "src"):
    shutil.copytree(PROJECT_ROOT / "tools" / "repo" / subdir, TRAIN_REPO / subdir)

patch_spec = importlib.util.spec_from_file_location("friend_a_train_patches", PATCH_SOURCE)
if patch_spec is None or patch_spec.loader is None:
    raise RuntimeError(f"Cannot import training patches from {PATCH_SOURCE}")
patch_module = importlib.util.module_from_spec(patch_spec)
patch_spec.loader.exec_module(patch_module)
TRAIN_SCRIPT = TRAIN_REPO / "scripts" / "train_unet_transformer.py"
patch_module.apply_train_patches(TRAIN_SCRIPT)
trainer_source = TRAIN_SCRIPT.read_text(encoding="utf-8")
legacy_models_import = "from biohub_tracking.models import SimpleNodeTransformer, TemporalUNet3D"
if legacy_models_import in trainer_source:
    trainer_source = trainer_source.replace(
        legacy_models_import,
        "from biohub_tracking.models.simple_node_transformer import SimpleNodeTransformer\n"
        "from biohub_tracking.models.temporal_unet import TemporalUNet3D",
        1,
    )
    TRAIN_SCRIPT.write_text(trainer_source, encoding="utf-8")
    print("patched local namespace-package model imports")
compile(TRAIN_SCRIPT.read_text(encoding="utf-8"), str(TRAIN_SCRIPT), "exec")

help_env = {
    **os.environ,
    "PYTHONPATH": str(TRAIN_REPO / "src"),
    "BIOHUB_DATA_DIR": str(DATA_DIR),
    "PYTHONUTF8": "1",
    "PYTHONIOENCODING": "utf-8",
}
help_check = subprocess.run(
    [sys.executable, str(TRAIN_SCRIPT), "--help"],
    cwd=TRAIN_REPO,
    env=help_env,
    text=True,
    encoding="utf-8",
    errors="replace",
    capture_output=True,
)
if help_check.returncode != 0:
    raise RuntimeError("Patched trainer import/CLI check failed:\n" + (help_check.stdout + help_check.stderr)[-4000:])
for required_flag in ("--amp", "--no-grad-ckpt", "--seed", "--det-loss"):
    if required_flag not in help_check.stdout:
        raise RuntimeError(f"Patched trainer is missing required flag: {required_flag}")

split_for_trainer = [{"split": 0, "train": train_ids, "test": holdout_ids}]
TRAIN_SPLIT = RUN_DIR / "loeo_train6bba_val44b6_for_trainer.json"
TRAIN_SPLIT.write_text(json.dumps(split_for_trainer, ensure_ascii=False, indent=2), encoding="utf-8")

TRAIN_CMD = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--method", METHOD,
    "--data-dir", str(DATA_DIR),
    "--splits", str(TRAIN_SPLIT),
    "--split", "0",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--lr", str(LEARNING_RATE),
    "--det-loss-weight", str(DET_LOSS_WEIGHT),
    "--det-neg-weight", str(DET_NEG_WEIGHT),
    "--window-size", str(WINDOW_SIZE),
    "--downsample", DOWNSAMPLE,
    "--pool-kernel-um", str(POOL_KERNEL_UM),
    "--det-loss", DET_LOSS,
    "--seed", str(SEED),
    "--amp",
    "--no-grad-ckpt",
    "--single-gpu",
]
if MAX_ITERS is not None:
    TRAIN_CMD.extend(["--max-iters", str(MAX_ITERS)])
if AUG_INTENSITY:
    TRAIN_CMD.append("--aug-intensity")

for forbidden_flag in ("--init-weights", "--unet-weights"):
    if forbidden_flag in TRAIN_CMD:
        raise RuntimeError(f"Leakage guard: forbidden scratch-training flag found: {forbidden_flag}")

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL
    ).strip()
except Exception:
    git_commit = None

RUN_CONFIG = {
    "experiment": EXPERIMENT,
    "run_id": RUN_ID,
    "status": "prepared",
    "created_at": datetime.now().astimezone().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "git_commit": git_commit,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_device": device_name,
    "train_embryo": "6bba",
    "validation_embryo": "44b6",
    "train_videos": train_ids,
    "validation_videos": holdout_ids,
    "split_sha256": hashlib.sha256(SPLIT_SOURCE.read_bytes()).hexdigest(),
    "initialization": "scratch",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
    "window_size": WINDOW_SIZE,
    "det_loss": DET_LOSS,
    "det_neg_weight": DET_NEG_WEIGHT,
    "det_loss_weight": DET_LOSS_WEIGHT,
    "downsample": DOWNSAMPLE,
    "pool_kernel_um": POOL_KERNEL_UM,
    "aug_intensity": AUG_INTENSITY,
    "max_iters_per_epoch": MAX_ITERS,
    "amp": True,
    "gradient_checkpointing": False,
    "command": TRAIN_CMD,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "command.txt").write_text(subprocess.list2cmdline(TRAIN_CMD) + "\n", encoding="utf-8")
(RUN_DIR / "results.json").write_text(
    json.dumps(
        {
            "experiment": EXPERIMENT,
            "run_id": RUN_ID,
            "status": "prepared",
            "official_metric": None,
            "note": "Official prediction/evaluation is Friend B's follow-up task.",
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("scratch command prepared (no pretrained weights):")
print(subprocess.list2cmdline(TRAIN_CMD))


In [ ]:
# Stream the long-running process to the notebook and to train.log.

import re


def write_results(payload: dict) -> None:
    (RUN_DIR / "results.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


if RUN_TRAINING:
    started_at = datetime.now().astimezone()
    train_log = RUN_DIR / "train.log"
    trainer_env = {
        **os.environ,
        "BIOHUB_DATA_DIR": str(DATA_DIR),
        "PYTHONPATH": str(TRAIN_REPO / "src"),
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    }
    print(f"Training started at {started_at.isoformat()}", flush=True)
    with train_log.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            TRAIN_CMD,
            cwd=TRAIN_REPO,
            env=trainer_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="", flush=True)
                log_handle.write(line)
                log_handle.flush()
            return_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait(timeout=30)
            write_results(
                {
                    "experiment": EXPERIMENT,
                    "run_id": RUN_ID,
                    "status": "interrupted",
                    "official_metric": None,
                    "log": str(train_log),
                }
            )
            raise

    finished_at = datetime.now().astimezone()
    wall_seconds = (finished_at - started_at).total_seconds()
    if return_code != 0:
        write_results(
            {
                "experiment": EXPERIMENT,
                "run_id": RUN_ID,
                "status": "failed",
                "return_code": return_code,
                "official_metric": None,
                "wall_seconds": wall_seconds,
                "log": str(train_log),
            }
        )
        raise RuntimeError(f"Training failed with exit code {return_code}; see {train_log}")

    checkpoint_source = TRAIN_REPO / "weights" / METHOD / "split_0" / "edge_predictor_best.pth"
    model_config_source = checkpoint_source.parent / "config.json"
    if not checkpoint_source.exists() or not model_config_source.exists():
        raise FileNotFoundError("Training ended successfully but the expected checkpoint/config pair is missing")

    ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
    checkpoint_target = ARTIFACT_DIR / "edge_predictor_best.pth"
    model_config_target = ARTIFACT_DIR / "config.json"
    shutil.copy2(checkpoint_source, checkpoint_target)
    shutil.copy2(model_config_source, model_config_target)
    shutil.copy2(RUN_DIR / "run_config.json", ARTIFACT_DIR / "run_config.json")

    log_text = train_log.read_text(encoding="utf-8", errors="replace")
    epoch_pattern = re.compile(
        r"Epoch\s+(?P<epoch>\d+)/(?P<epochs>\d+).*?edge=(?P<edge>[0-9.eE+-]+).*?"
        r"det=(?P<det>[0-9.eE+-]+).*?test_loss=(?P<test_loss>[0-9.eE+-]+).*?"
        r"acc=(?P<acc>[0-9.eE+-]+).*?recall=(?P<recall>[0-9.eE+-]+).*?best=(?P<best>[0-9.eE+-]+)"
    )
    epoch_rows = []
    for match in epoch_pattern.finditer(log_text):
        row = {"epoch": int(match.group("epoch"))}
        row.update({key: float(match.group(key)) for key in ("edge", "det", "test_loss", "acc", "recall", "best")})
        epoch_rows.append(row)

    checkpoint_sha256 = hashlib.sha256(checkpoint_target.read_bytes()).hexdigest()
    results = {
        "experiment": EXPERIMENT,
        "run_id": RUN_ID,
        "status": "trained_not_officially_scored",
        "started_at": started_at.isoformat(),
        "finished_at": finished_at.isoformat(),
        "wall_seconds": wall_seconds,
        "train_videos": len(train_ids),
        "validation_videos": len(holdout_ids),
        "validation_set": "LOEO-44b6",
        "initialization": "scratch",
        "checkpoint": str(checkpoint_target),
        "checkpoint_sha256": checkpoint_sha256,
        "model_config": str(model_config_target),
        "internal_sparse_validation": epoch_rows,
        "official_metric": None,
        "note": "Internal acc/recall is diagnostic only; Friend B must run prediction and the official metric.",
    }
    write_results(results)
    shutil.copy2(RUN_DIR / "results.json", ARTIFACT_DIR / "results.json")
    print(f"Training complete in {wall_seconds / 3600:.2f} h")
    print(f"Hand-off artifact: {ARTIFACT_DIR}")
else:
    print("BIOHUB_RUN_TRAINING=0: preflight passed; training was intentionally skipped.")


In [ ]:
# Verify and describe the exact hand-off to Friend B/Vitalii. This cell never uploads or submits anything.

results_path = RUN_DIR / "results.json"
results = json.loads(results_path.read_text(encoding="utf-8"))

if results["status"] == "trained_not_officially_scored":
    required_artifacts = [
        ARTIFACT_DIR / "edge_predictor_best.pth",
        ARTIFACT_DIR / "config.json",
        ARTIFACT_DIR / "run_config.json",
        ARTIFACT_DIR / "results.json",
    ]
    missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
    if missing_artifacts:
        raise FileNotFoundError("Incomplete hand-off artifact:\n" + "\n".join(missing_artifacts))

    handoff = {
        "experiment": EXPERIMENT,
        "status": "ready_for_friend_b_evaluation",
        "weights_dataset_subfolder": EXPERIMENT,
        "files_for_weights_dataset": [
            str(ARTIFACT_DIR / "edge_predictor_best.pth"),
            str(ARTIFACT_DIR / "config.json"),
        ],
        "supporting_files": [
            str(ARTIFACT_DIR / "run_config.json"),
            str(ARTIFACT_DIR / "results.json"),
            str(RUN_DIR / "train.log"),
        ],
        "evaluation_request": {
            "sets": ["LOEO-44b6", "48 (hold12 + divset36)"],
            "metric": "official competition metric",
            "leaderboard_submission": False,
        },
    }
    handoff_path = RUN_DIR / "handoff.json"
    handoff_path.write_text(json.dumps(handoff, ensure_ascii=False, indent=2), encoding="utf-8")
    shutil.copy2(handoff_path, ARTIFACT_DIR / "handoff.json")
    print("READY FOR HAND-OFF")
    print(f"Send this directory to Friend B/Vitalii: {ARTIFACT_DIR}")
    print(f"Then copy the measured result into experiments/{EXPERIMENT}/results.json and RESULTS.md.")
else:
    print(f"Current run status: {results['status']}. No checkpoint hand-off was created.")
